In [ ]:
from services.yt_comments import func_get_comments

In [56]:
import re
import pandas as pd
from sentence_transformers import SentenceTransformer

In [57]:
df = func_get_comments('AtT0MorhYE4')

2025-11-15 17:11:19,883 - root - INFO - Loading comments from cache: AtT0MorhYE4
2025-11-15 17:11:19,892 - root - INFO - Parsed 4502 comments from response and converted to dataframe.
2025-11-15 17:11:19,898 - root - INFO - Comment Extraction [AtT0MorhYE4]: Time taken: 0.01s


In [58]:
df.head()

,author,comment,likes,published_at
0,@samiranroyy1700,PLEASE GIVE ME THE MASHUP CHORD PROGRESSIONS,None,2025-11-15T08:20:42Z
1,@movieediting3117,This is my dream to sit beside him and sing so...,None,2025-11-15T06:44:20Z
2,@umeshbhati4u,Chitoye.. tum ro hi lo.. ha tu hai ha tu hai.....,None,2025-11-15T00:12:47Z
3,@umeshbhati4u,Could have been much better,None,2025-11-15T00:10:35Z
4,@pavanthorat1595,Sorry to say but those who are singing looking...,None,2025-11-14T19:11:51Z


In [59]:
raw_comments = df['comment'].tolist()

In [60]:
def preprocess_comments(text:str) -> str:
    if not isinstance(text, str):
        return ""
    
    # remove urls
    text = re.sub(r'https\S+|www\.\S+', "", text)

    # lowercase
    text = text.lower()

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text 

In [61]:
raw_comments

['PLEASE GIVE ME THE MASHUP CHORD PROGRESSIONS',
 'This is my dream to sit beside him and sing songs like this❤️🥰',
 'Chitoye.. tum ro hi lo.. ha tu hai ha tu hai.. kya bakwass kar diya.. tumhai accha lag raha hoga.. maal fook.. kai.. par hai hai nahi itna achha..',
 'Could have been much better',
 'Sorry to say but those who are singing looking at the papers in their hands were not deserved to be in the video\nWake me from my sleep and i can sing any of the emraan hashmi song ....anytime..\n\nAkhha bollywood ek taraf, emran hashmi ek tarah',
 'OMG..!! This is simply Fantastic.. Awesome.. Heart Touching. Lovely.. Beautiful..\nHats Off.... Sajdaa.... Sajdaa....🙏♥🌹♥🌹♥🌹♥👌',
 'Lovely songs 👌✨👍',
 '♥♥♥♥',
 'an average actor gets most of the credit for all these amazing songs more than the singers even.\nthats bollywood for u',
 'He surely miss legend KK during whole performance ❤\nAkhi duniya ek taraf Aur Emraan Hashmi and kk ka combo ek taraf',
 '3:12 that shine in his eyes',
 'Woh gujju b

In [62]:
cleaned_comments  = [preprocess_comments(c) for c in raw_comments]

In [63]:
cleaned_comments

['please give me the mashup chord progressions',
 'this is my dream to sit beside him and sing songs like this❤️🥰',
 'chitoye.. tum ro hi lo.. ha tu hai ha tu hai.. kya bakwass kar diya.. tumhai accha lag raha hoga.. maal fook.. kai.. par hai hai nahi itna achha..',
 'could have been much better',
 'sorry to say but those who are singing looking at the papers in their hands were not deserved to be in the video wake me from my sleep and i can sing any of the emraan hashmi song ....anytime.. akhha bollywood ek taraf, emran hashmi ek tarah',
 'omg..!! this is simply fantastic.. awesome.. heart touching. lovely.. beautiful.. hats off.... sajdaa.... sajdaa....🙏♥🌹♥🌹♥🌹♥👌',
 'lovely songs 👌✨👍',
 '♥♥♥♥',
 'an average actor gets most of the credit for all these amazing songs more than the singers even. thats bollywood for u',
 'he surely miss legend kk during whole performance ❤ akhi duniya ek taraf aur emraan hashmi and kk ka combo ek taraf',
 '3:12 that shine in his eyes',
 'woh gujju bhai jo 

In [64]:
def chunk_comments(comments, max_words = 200):
    chunks = []
    current_chunk = []
    word_count = 0

    for comment in comments:
        words = comment.split()

        if len(words) + word_count > max_words:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            word_count = 0
        
        current_chunk.append(comment)
        word_count += len(words)
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks

In [65]:
chunks = chunk_comments(cleaned_comments)
chunks

['please give me the mashup chord progressions this is my dream to sit beside him and sing songs like this❤️🥰 chitoye.. tum ro hi lo.. ha tu hai ha tu hai.. kya bakwass kar diya.. tumhai accha lag raha hoga.. maal fook.. kai.. par hai hai nahi itna achha.. could have been much better sorry to say but those who are singing looking at the papers in their hands were not deserved to be in the video wake me from my sleep and i can sing any of the emraan hashmi song ....anytime.. akhha bollywood ek taraf, emran hashmi ek tarah omg..!! this is simply fantastic.. awesome.. heart touching. lovely.. beautiful.. hats off.... sajdaa.... sajdaa....🙏♥🌹♥🌹♥🌹♥👌 lovely songs 👌✨👍 ♥♥♥♥ an average actor gets most of the credit for all these amazing songs more than the singers even. thats bollywood for u he surely miss legend kk during whole performance ❤ akhi duniya ek taraf aur emraan hashmi and kk ka combo ek taraf 3:12 that shine in his eyes woh gujju bhai jo lead kar raha tha, itna besura, imraan se bh

In [66]:
len(chunks)

228

In [67]:
embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

2025-11-15 17:11:20,121 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2025-11-15 17:11:20,122 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


In [ ]:
def embedding_chunks(chunks):
    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True,
        batch_size=32
        )
    print(embeddings.shape)

    similarities = embedding_model.similarity(embeddings, embeddings)
    print(similarities)
    return embeddings

In [69]:
yt_embeddings = embedding_chunks(chunks)

Batches: 100%|██████████| 8/8 [00:51<00:00,  6.48s/it]

(228, 768)
tensor([[1.0000, 0.7343, 0.6907,  ..., 0.7381, 0.6043, 0.7140],
        [0.7343, 1.0000, 0.7174,  ..., 0.6773, 0.4901, 0.6975],
        [0.6907, 0.7174, 1.0000,  ..., 0.6354, 0.4856, 0.6059],
        ...,
        [0.7381, 0.6773, 0.6354,  ..., 1.0000, 0.6665, 0.7783],
        [0.6043, 0.4901, 0.4856,  ..., 0.6665, 1.0000, 0.6796],
        [0.7140, 0.6975, 0.6059,  ..., 0.7783, 0.6796, 1.0000]])


In [70]:
(yt_embeddings.shape)[1]

768

In [72]:
class VectorDB:
    def __init__(self,embedding_dim:int):
        '''
        Embedding_dim  = size of each embedding (number of features used to represent the meaning)
        [768 for mpnet]
        '''

        self.embedding_dim = embedding_dim
        self.embeddings = np.empty((0,embedding_dim), dtype='float32')
        self.chunks = [] # text storage
        self.ids = [] # simple incremental IDS

        self.__next_id = 0
    
    def add(self, chunk:str, embedding: np.array):
        """
        Insert a single chunk + embedding 
        

        """

        if embedding.shape[0] != self.embedding_dim:
            raise ValueError("Embedding dimension mismatch")
        
        # add text
        self.chunks.append(chunk)

        # add embedding --- shape of input embedding = (768,)
        embedding = embedding.reshape(1,-1)
        # after reshape --- (1,768)

        self.embeddings = np.vstack([self.embeddings, embedding])

        # add ID
        new_id = self.__next_id
        self.ids.append(new_id)

        self.__next_id += 1

        return new_id

    def add_all(self, chunk_list, embedding_list):
        '''
        Insert multiple chunks + embedding in batch 
        '''
        # convert the embedding list to numpy
        embeddings_list = np.array(embedding_list, dtype="float32")

        if embeddings_list.shape[1] != self.embedding_dim:
            raise ValueError("Embedding dimension mismatch")
        
        # add all chunks
        self.chunks.extend(chunk_list)
        # add all embeddings at once
        self.embeddings = np.vstack([self.embeddings, embedding_list])

        # create ids for this batch
        start_id = self.__next_id
        end_id = start_id + len(chunk_list)

        batch_ids = list(range(start_id,end_id))
        self.ids.extend(batch_ids)

        self.__next_id = end_id

        return batch_ids
    
    def _cosine_similarity(self, query_vec: np.ndarray, matrix: np.ndarray):
        """
        Compute similarity between query vector and all stored embeddings.
        """
        # normalize query
        query_norm = query_vec / (np.linalg.norm(query_vec) + 1e-10)
        # normalize all embeddings
        matrix_norm = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True)+ 1e-10)

        scores = np.dot(matrix_norm, query_norm)

        return scores # shape: (num_chunks, ) a list of num_chunks which is 227
    
    def search(self, query_embedding: np.ndarray, top_k: int=5):
        """
        Find top k most similar chunks to the query embedding
        Return: list of dict{id, chunk, score}
        """
        if query_embedding.shape[0] != self.embedding_dim:
            raise ValueError("Embedding dimensions mismatch")

        # compute cosine similarity
        scores = self._cosine_similarity(query_embedding, self.embeddings)

        # get top-k indexes in descending order
        top_idx = np.argsort(scores)[::-1][:top_k]

        results = []

        for idx in top_idx:
            results.append({
                "id": self.ids[idx],
                "chunk": self.chunks[idx],
                "score": float(scores[idx])
            })
        
        return results
    
    def save(self,folder_path:str):
        """
        Save embeddings, chunks and metadata to disk 
        """

        os.makedirs(folder_path, exist_ok=True)

        # save emebddings
        np.save(os.path.join(folder_path, "embeddings.npy"), self.embeddings)

        # save chunks + id + metadata
        metadata = {
            "chunks" : self.chunks,
            "ids" : self.ids,
            "embedding_dim": self.embedding_dim,
            "next_id": self.__next_id
        }

        with open(os.path.join(folder_path, "metadata.json"), "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=4)
        
        return True
    
    @classmethod
    def load(cls, folder_path: str):
        """
        load the disk
        """
        
        # load metadata
        with open(os.path.join(folder_path, "metadata.json"), 'r', encoding="utf-8") as f:
            metadata = json.load(f)
        
        # create new instance [ cls == Vectordb]
        db = cls(embedding_dim = metadata['embedding_dim'])

        # load embeddings
        db.embeddings = np.load(os.path.join(folder_path, "embeddings.npy"))

        # load chunks + ids
        db.chunks = metadata['chunks']
        db.ids = metadata['ids']
        db.__next_id = metadata['next_id']

        return db


In [73]:
dim = yt_embeddings.shape[1]

In [74]:
db = VectorDB(dim)

In [75]:
batch_ids = db.add_all(chunks, yt_embeddings)

In [82]:
user_query = "Did they love imraan hasmi in video"
user_query_embedding = embedding_chunks(user_query)

Batches: 100%|██████████| 1/1 [00:00<00:00, 16.90it/s]

(768,)
tensor([[1.0000]])


In [83]:
results = db.search(user_query_embedding, 10)

In [84]:
results

[{'id': 14,
  'chunk': "this is the 100million view video..... ..from a super fan of great lover, great actor, great justifier of song... aura hay bhai ka, love from bangladesh 🇧🇩 akka bollywood ek trf hashmi ek trf ❤❤26sep aaj kon kon dekha akkha bollywood ek taraf aur emraan hasmi ek taraf bc 😍😍 90s people enjoying and few gen-z guys standing clueless😂😂😂 everyone vibing except that one girl on the left who has no clue why she's there! 😅 cord please? emran hashmi muje bhi ganna sikhna chahiye 😂 made my heart full <3 neha kakkar aur tanishq bagchi ye sunenge to sharm me mar jaayenge what a mad.. mad... mad... madness !! ❤ ye gaane nahi khajaana hai ❤ i lived my childhood in these 7 minutes 🥺❤️ a big big thanks to entire team and our all time favorite imran hashmi kk + emran hashmi= 🔥 superb ❤❤❤ this man is🫶❤️ emraan hasmi paglu present ! now i remember what i did in my childhood 🤣",
  'score': 0.6249017119407654},
 {'id': 148,
  'chunk': "why are gangs of ladies far from imran 🤔🤔 man! 